# Project A — Closeout: end-to-end validation of the research substrate

**Task**: `A-CLOSE` (`docs/PRIORITIES.yaml`). **Methodology**: `docs/METHODOLOGY.md`
§21 (project closeout) — the project-scale analog of §17's per-change E2E notebook
and §20's post-task review.

**Why this notebook exists.** Project A's individual tasks each passed their own
unit tests and §20 review *in isolation*. Nothing so far validates that the
assembled substrate — feature catalog, priorities drift test, trial-count
ledger, runner->ledger integration, and the DSR-aware gate — actually *composes*
across the module boundaries those tasks created. This notebook is that
integration seam: it exercises the real, shipped surface end-to-end and asserts
every seam is green.

It is a **checkpoint-only** notebook (METHODOLOGY §7): it fits no model. The
DSR-aware gate runs on the Phase 4A arm checkpoints written by
`scripts/run_phase4a_arms.py`, so the closeout verdict is reproducible in
seconds and re-derives the known Phase 4A NO-GO from the assembled machinery.

**Sections**

1. Feature catalog contract — `features/catalog.py` + `catalog.yaml` (§6 drift)
2. Priorities drift test — `tests/test_priorities.py` (§6 backlog integrity)
3. Trial-count ledger read — `ledger.py` + `data/ledger.yaml` (§12)
4. Runner->ledger integration — `record_run` on a temp ledger (§12, A-LEDGER-RUNNERS)
5. DSR-aware gate on real checkpoints — `regime_metrics.py` (§13, A-DSR-GATE)
6. Closeout verdict — all seams asserted green


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import importlib.util
import shutil
import sys
import tempfile
from pathlib import Path
from types import SimpleNamespace

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd

# One place to accumulate seam verdicts; the final cell asserts they are all True.
SEAMS = {}

print(f'numpy   {np.__version__}')
print(f'pandas  {pd.__version__}')
print(f'repo    {ROOT}')

---
## 1 — Feature catalog contract (`features/catalog.py` + `catalog.yaml`)

The catalog is the code-vs-config contract M4 built for features (METHODOLOGY
§4/§6). `validate_catalog_coverage` asserts `set(produced) == set(registered)`
and names the offender in **both** directions (`unregistered`, `phantom`). Here
we load the real 27-record catalog, show it validates against its own name set,
and confirm an injected unregistered column is caught — the drift contract works
both ways.

In [ ]:
from quant.features.catalog import load_catalog, validate_catalog_coverage

catalog = load_catalog()
print(f'catalog loaded: {len(catalog)} feature records')

# Forward direction: the catalog's own name set is internally consistent.
validate_catalog_coverage(set(catalog.keys()), catalog)
print('validate_catalog_coverage(catalog names) -> OK (no unregistered / phantom)')

# Reverse direction: an unregistered column must be named and raise (METHODOLOGY §6).
drift_caught = False
try:
    validate_catalog_coverage(set(catalog.keys()) | {'phantom_feature_xyz'}, catalog)
except ValueError as exc:
    drift_caught = True
    print(f'injected drift correctly raised: {str(exc)[:90]}...')

SEAMS['catalog_contract'] = (len(catalog) == 27) and drift_caught
print(f"\nseam[catalog_contract] = {SEAMS['catalog_contract']}")

---
## 2 — Priorities drift test (`tests/test_priorities.py`)

The backlog's own integrity contract (A-PRIORITIES-TEST + A-PRIORITIES-TEST-TS):
status enum validity, dependency-reference resolution, at-most-one `in_progress`,
`started_at`/`completed_at` presence and ordering. We import the drift-test
module by path (`tests/` is not a package) and run its top-level
`validate_priorities` against the committed file — the same check CI runs.

In [ ]:
spec = importlib.util.spec_from_file_location(
    'test_priorities', ROOT / 'tests' / 'test_priorities.py'
)
tp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tp)

data = tp.load_priorities()
tp.validate_priorities(data)  # raises ValueError naming the offender on any drift

tasks = data['tasks']
by_status = {}
for t in tasks:
    by_status[t['status']] = by_status.get(t['status'], 0) + 1
print(f'validate_priorities(PRIORITIES.yaml) -> OK ({len(tasks)} tasks)')
print('status histogram:', dict(sorted(by_status.items())))

# The invariant that makes 'pick the next ready task' well-defined: exactly one in_progress.
one_in_progress = by_status.get('in_progress', 0) == 1
SEAMS['priorities_drift'] = one_in_progress
print(f"\nseam[priorities_drift] = {SEAMS['priorities_drift']} (exactly one in_progress: {one_in_progress})")

---
## 3 — Trial-count ledger read (`ledger.py` + `data/ledger.yaml`)

The append-only trial-count ledger (METHODOLOGY §12). Its cumulative `N` (sum of
`n_comparisons`) is the Bailey-Lopez de Prado deflation count the DSR-aware gate
reads in §5 — it must come from this file, not hand-counting.
`observed_sharpe_std` supplies the empirical cross-trial Sharpe dispersion when
>=2 entries carry a `sharpe`; today none do, so it returns `None` and the gate
falls back to the pinned `DEFAULT_SHARPE_STD` (demonstrated in §5).

In [ ]:
from quant.ledger import (
    cumulative_trial_count,
    load_ledger,
    observed_sharpe_std,
    record_run,
)

entries = load_ledger()
N = cumulative_trial_count(entries)
emp_std = observed_sharpe_std(entries)
print(f'ledger entries: {len(entries)}')
print(f'cumulative N (sum n_comparisons): {N}')
print(f'observed_sharpe_std: {emp_std}  '
      '(None -> too few sharpe-bearing entries; gate uses pinned DEFAULT_SHARPE_STD)')
print('\nmost recent 5 entries:')
for e in entries[-5:]:
    print(f'  {e.id}  {e.prd}/{e.milestone}  n={e.n_comparisons}  verdict={e.verdict}')

SEAMS['ledger_read'] = (len(entries) > 0) and (N == sum(e.n_comparisons for e in entries))
print(f"\nseam[ledger_read] = {SEAMS['ledger_read']}")

---
## 4 — Runner->ledger integration (`record_run`)  ·  A-LEDGER-RUNNERS

METHODOLOGY §12 asks the ledger to be *"written by every runner"*, not populated
by hand. `record_run` maps a runner's `metadata.json` (as written by
`scripts/run_phase4a_arms.py`) to a validated `LedgerEntry` and appends it,
idempotently by `config_hash`.

**Honest declaration (METHODOLOGY §9).** The real `data/ledger.yaml` is
append-only and CI-audited; appending a synthetic demo trial to it would pollute
the audited artifact. So this cell runs `record_run` against a **temp copy** of
the ledger — proving the runner->ledger seam works and increments `N`, and that a
re-run with the same `config_hash` is a no-op — while the committed ledger stays
untouched.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    tmp_ledger = Path(td) / 'ledger.yaml'
    shutil.copy(ROOT / 'data' / 'ledger.yaml', tmp_ledger)

    n_before = cumulative_trial_count(path=tmp_ledger)

    # Synthetic runner metadata — same shape run_phase4a_arms.py writes.
    synth_meta = {
        'config_hash': 'aclose-demo-' + '0' * 52,
        'started_at': '2026-07-12T00:00:00Z',
        'finished_at': '2026-07-12T00:05:00Z',
        'aggregate_sharpe': 0.31,
    }
    entry = record_run(
        synth_meta,
        prd='project-a-closeout',
        milestone='A-CLOSE',
        preregistration='docs/PROJECT_A_CLOSEOUT.md',
        n_comparisons=3,
        verdict='inconclusive',
        agent='human',
        notes='A-CLOSE closeout demo trial (temp ledger; NOT the audited file).',
        path=tmp_ledger,
    )
    n_after = cumulative_trial_count(path=tmp_ledger)
    print(f'appended {entry.id}: n_comparisons={entry.n_comparisons}, sharpe={entry.sharpe}')
    print(f'N: {n_before} -> {n_after}  (delta = {n_after - n_before})')

    # Idempotency: same config_hash is a no-op (re-running a runner never double-counts N).
    again = record_run(
        synth_meta, prd='project-a-closeout', milestone='A-CLOSE',
        preregistration='docs/PROJECT_A_CLOSEOUT.md', n_comparisons=3,
        verdict='inconclusive', path=tmp_ledger,
    )
    n_final = cumulative_trial_count(path=tmp_ledger)
    print(f're-run same config_hash -> {"no-op (None)" if again is None else again.id}; N stays {n_final}')

incremented = (n_after - n_before) == 3
idempotent = (again is None) and (n_final == n_after)
# The audited file is unchanged: its N still equals the §3 read.
untouched = cumulative_trial_count() == N
SEAMS['runner_ledger'] = incremented and idempotent and untouched
print(f'\nreal data/ledger.yaml untouched (N still {cumulative_trial_count()}): {untouched}')
print(f"seam[runner_ledger] = {SEAMS['runner_ledger']}")

---
## 5 — DSR-aware gate on real Phase 4A checkpoints (`regime_metrics.py`)  ·  A-DSR-GATE

The integration climax. `dsr_aware_gate_report` is the two-stage gate
(METHODOLOGY §13): **stage 1** is the regime Sharpe/DM gate (`phase4a_gate_report`
verbatim); **stage 2** deflates the GBM aggregate Sharpe against the ledger's
cumulative `N` (Bailey-Lopez de Prado). It reads `N` from §3's ledger
automatically.

We load the real `arima` (control) and `signed` (GBM) checkpoints, align to their
common OOS index, tag eras with `DateRangeDetector`, and call the **real** gate
function. The gate reads only `.oos_returns` / `.oos_forecast_errors`, so
checkpoint-derived `SimpleNamespace`s are sufficient (duck typing — no model
refit). The verdict should reproduce the Phase 4A NO-GO
(`docs/PHASE_4A_REPORT.md`) from the assembled substrate.

In [ ]:
from quant.backtest.regimes import DateRangeDetector, tag_regimes
from quant.backtest.regime_metrics import DSR_THRESHOLD, dsr_aware_gate_report
from quant.backtest.statistics import DEFAULT_SHARPE_STD

CKPT = ROOT / 'data' / 'phase4a'


def _normalize_index(s):
    # Returns are NY-local, errors UTC from the harness; normalize to UTC tz-naive.
    if s.index.tz is not None:
        s = s.copy()
        s.index = s.index.tz_convert('UTC').tz_localize(None)
    return s


def load_arm(arm):
    base = CKPT / arm
    ret = _normalize_index(pd.read_parquet(base / 'oos_returns.parquet')['oos_returns'])
    err = _normalize_index(pd.read_parquet(base / 'oos_forecast_errors.parquet')['oos_forecast_errors'])
    return SimpleNamespace(oos_returns=ret, oos_forecast_errors=err)


arima = load_arm('arima')
signed = load_arm('signed')

idx = arima.oos_returns.index.intersection(signed.oos_returns.index)
for a in (arima, signed):
    a.oos_returns = a.oos_returns.loc[idx]
    a.oos_forecast_errors = a.oos_forecast_errors.loc[idx]

era = tag_regimes(idx, DateRangeDetector())
print(f'aligned OOS panel: {len(idx)} bars, {idx.min().date()} -> {idx.max().date()}')
print(f'pinned constants: DSR_THRESHOLD={DSR_THRESHOLD}, DEFAULT_SHARPE_STD={DEFAULT_SHARPE_STD}')

In [ ]:
report = dsr_aware_gate_report(signed, arima, era)

print('--- deflation inputs (read from the ledger, not hand-counted) ---')
print(f"  n_trials (ledger N):   {report['n_trials']}")
print(f"  sharpe_std:            {report['sharpe_std']:.4f}  (source={report['sharpe_std_source']})")
print()
print('--- stage 1: regime Sharpe / DM gate ---')
print(f"  stage1_passed:         {report['stage1_passed']}   pass_count={report['pass_count']}")
for r, m in report['per_regime'].items():
    dmp = m['dm_p_value']
    dmp_s = f'{dmp:.4f}' if dmp is not None else 'None'
    print(f"    {r:<11} gbm={m['gbm_sharpe']:+.3f}  arima={m['arima_sharpe']:+.3f}  "
          f"beats={m['gbm_beats_arima']}  dm_p={dmp_s}  n={m['n_bars']}")
print()
print('--- stage 2: deflated Sharpe (Bailey-Lopez de Prado) ---')
print(f"  sr_observed:           {report['sr_observed']:+.4f}")
print(f"  sr_benchmark (E[max]): {report['sr_benchmark']:+.4f}")
print(f"  dsr:                   {report['dsr']:.4f}   dsr_passed={report['dsr_passed']}")
print()
print(f"COMBINED gate_passed:    {report['gate_passed']}  "
      "(reproduces the Phase 4A NO-GO from the assembled substrate)")

# The gate must (a) consume N from the ledger and (b) return a coherent verdict.
gate_used_ledger_N = report['n_trials'] == cumulative_trial_count()
verdict_coherent = report['gate_passed'] == (report['stage1_passed'] and report['dsr_passed'])
SEAMS['dsr_gate'] = gate_used_ledger_N and verdict_coherent and (report['gate_passed'] is False)
print(f"\nseam[dsr_gate] = {SEAMS['dsr_gate']} "
      f"(gate read ledger N: {gate_used_ledger_N}; verdict coherent: {verdict_coherent})")

---
## 6 — Closeout verdict

Every module boundary Project A created has now been exercised against the real,
shipped artifacts. The assertion below is the notebook's definition-of-done: if
any seam regressed, `nbconvert --execute` fails here loudly rather than
producing a green-looking but hollow closeout.

In [ ]:
print('Project A substrate — integration seam verdicts:')
for name, ok in SEAMS.items():
    print(f'  [{"PASS" if ok else "FAIL"}]  {name}')

assert all(SEAMS.values()), f'Project A closeout FAILED: {[k for k, v in SEAMS.items() if not v]}'
print('\nALL SEAMS GREEN — Project A substrate composes end-to-end.')
print('See docs/PROJECT_A_CLOSEOUT.md for the one-page closeout report.')